# Diccionario de datos
Este notebook permite generar un diccionario de datos a partir de los 3 archivos CSV del INER: 

* `INER_COVID19_CostoPacientes_Econo.csv`
* `INER_COVID19_Pacientes_DiagnosticoComorbilidad.csv` 
* `INER_COVID19_TrabajoSocial.csv` 

In [ ]:
from pathlib import Path
ruta_base = Path().absolute().parent\
ruta_datos = ruta_base / 'data'
archivo_1 = 'INER_COVID19_CostoPacientes_Econo.csv'
archivo_2 = 'INER_COVID19_Pacientes_DiagnosticoComorbilidad.csv'
archivo_3 = 'INER_COVID19_TrabajoSocial.csv'

## 1. Generación Automática del Diccionario de Datos Exploratorio

Se utiliza la librería `Polars` para procesar eficientemente los archivos `.csv`. El script realiza las siguientes tareas:

1. **Detección de Encoding:** Identifica automáticamente la codificación de cada archivo para evitar errores de lectura utilizando `chardet`.
2. **Cálculo de Metadatos:** Analiza cada columna para obtener tipos de datos, conteo y porcentaje de nulos, valores únicos y un ejemplo de dato.
3. **Consolidación y Exportación:** Unifica la información de todos los archivos en un solo DataFrame (df_diccionario_maestro) y lo exporta como `01_Diccionario_Exploratorio.csv` para su consulta.

In [ ]:
import polars as pl
from pathlib import Path
import chardet # Para detectar encoding de archivos CSV

# 1. Configuración de rutas
base_path = Path(ruta_base)
archivos = [
    archivo_1,
    archivo_2,
    archivo_3
]

def obtener_encoding(ruta) -> str:
    """
    Detecta el encoding. Si chardet falla (retorna None),
    asume 'utf-8' por defecto para evitar romper Polars.
    """
    with open(ruta, 'rb') as f:
        rawdata = f.read(10000)

    detectado = chardet.detect(rawdata)['encoding']

    # Si detectado es None, retornamos 'utf-8' como fallback seguro
    return detectado if detectado is not None else 'utf-8'

def generar_diccionario_polars(archivo_nombre, ruta_base):
    ruta_completa = ruta_base / archivo_nombre
    encoding_detectado = obtener_encoding(ruta_completa)

    print(f"Procesando: {archivo_nombre} | Encoding: {encoding_detectado}")

    try:
        # Polars es estricto, si hay errores de parsing (comunes en CSVs sucios)
        # ignore_errors=True ayuda a leer lo que se pueda para el EDA
        df = pl.read_csv(ruta_completa, encoding=encoding_detectado, ignore_errors=True)

        filas_totales = df.height
        metadata = []

        for col_name in df.columns:
            col_series = df[col_name]
            nulos = col_series.null_count()

            metadata.append({
                "Archivo_Original": archivo_nombre,
                "Campo_Original": col_name,
                "Tipo_Dato_Detectado": str(col_series.dtype),
                "Total_Filas": filas_totales,
                "Cantidad_Nulos": nulos,
                "Porcentaje_Nulos": round((nulos / filas_totales) * 100, 2),
                "Unicos": col_series.n_unique(),
                # Tomamos el primer valor no nulo como ejemplo
                "Ejemplo": col_series.drop_nulls().head(1)[0] if col_series.drop_nulls().len() > 0 else None
            })

        return pl.DataFrame(metadata)

    except Exception as e:
        print(f" Error leyendo {archivo_nombre}: {e}")
        return None

# 2. Ejecución masiva
dfs_info = []
for archivo in archivos:
    res = generar_diccionario_polars(archivo, base_path)
    if res is not None:
        dfs_info.append(res)

# 3. Consolidación y Visualización
if dfs_info:
    df_diccionario_maestro = pl.concat(dfs_info)

    # Ajustar configuración para ver todo en Jupyter
    pl.Config.set_tbl_rows(100)
    print("\n--- DICCIONARIO DE DATOS EXPLORATORIO ---")
    display(df_diccionario_maestro) # Display sirve para mostrar en Jupyter de forma amigable

    # 4. Guardar el diccionario en CSV
    df_diccionario_maestro.write_csv("01_Diccionario_Exploratorio.csv")

## 2. Análisis de problemas de calidad de datos (limpieza necesaria)

**Generación del Plan de Limpieza**
Este bloque de código realiza las siguientes tareas automáticas:

1. Carga el diccionario exploratorio base (`01_Diccionario_Exploratorio.csv`).

2. Detecta Fechas Disfrazadas: Busca campos con `FECHA` o `NACIMIENTO` que Polars detectó como String y sugiere parseo.

3. Detecta Edad Compleja: Identifica si el campo EDAD viene como texto y sugiere extracción con Regex.

4. Detecta Suciedad en Texto: Revisa si el ejemplo tiene espacios al inicio/final o si el formato de mayúsculas es inconsistente.

5. Genera el CSV: Crea el archivo `02_Plan_Limpieza.csv` con estas sugerencias listas para que se valide o corrijas de forma manual, reduciendo el tiempo de limpieza manual.

In [ ]:
import polars as pl

# 1. Cargar el Diccionario Exploratorio (Base del conocimiento)
# Asumimos que este archivo ya existe tras ejecutar el paso anterior
df_exploratorio = pl.read_csv("01_Diccionario_Exploratorio.csv")

print(f"Diccionario base cargado: {df_exploratorio.height} campos a analizar.")

# 2. Definir Reglas Heurísticas (Lógica de Negocio Automática)
# Usamos pl.when().then() encadenados para priorizar problemas más graves

df_limpieza = df_exploratorio.with_columns([
    # A. Detección de Problemas de Calidad
    pl.when(
        (pl.col("Campo_Original").str.contains("(?i)FECHA|NACIMIENTO|INGRESO|EGRESO")) &
        (pl.col("Tipo_Dato_Detectado") == "String")
    ).then(pl.lit("Formato incorrecto (Texto en lugar de Fecha)"))

    .when(
        (pl.col("Campo_Original").str.contains("(?i)EDAD")) &
        (pl.col("Tipo_Dato_Detectado") == "String")
    ).then(pl.lit("Edad en texto libre/mixto"))

    .when(
        (pl.col("Tipo_Dato_Detectado") == "String") &
        (pl.col("Ejemplo").str.contains(r"^\s|\s$")) # Espacios al inicio o final
    ).then(pl.lit("Espacios en blanco redundantes (suciedad)"))

    .otherwise(pl.lit("Aparentemente correcto / Revisión manual"))
    .alias("Problema_Calidad"),

    # B. Sugerencia de Reglas de Limpieza (Tratamiento)
    pl.when(
        (pl.col("Campo_Original").str.contains("(?i)FECHA|NACIMIENTO|INGRESO|EGRESO")) &
        (pl.col("Tipo_Dato_Detectado") == "String")
    ).then(pl.lit("Parseo a Date ISO-8601 (YYYY-MM-DD)"))

    .when(
        (pl.col("Campo_Original").str.contains("(?i)EDAD")) &
        (pl.col("Tipo_Dato_Detectado") == "String")
    ).then(pl.lit("Extracción numérica (Regex) y cast a Int"))

    .when(
        (pl.col("Tipo_Dato_Detectado") == "String")
    ).then(pl.lit("Trim espacios + Estandarizar a Mayúsculas + Quitar acentos"))

    .otherwise(pl.lit("Mantener formato original"))
    .alias("Regla_Limpieza")
])

# 3. Seleccionar solo las columnas relevantes para el Plan de Limpieza
# (Omitimos conteos de nulos/unicos que van en otros reportes)
df_limpieza_export = df_limpieza.select([
    "Archivo_Original",
    "Campo_Original",
    "Tipo_Dato_Detectado",
    "Ejemplo",
    "Problema_Calidad",
    "Regla_Limpieza"
])

# 4. Guardar
output_file = "02_Plan_Limpieza.csv"
df_limpieza_export.write_csv(output_file)

print(f" Archivo generado: {output_file}")
print("   Sugerencias automáticas aplicadas: Revisión manual a continuación.")
display(df_limpieza_export)

## 3. Análisis de valores nulos

**Generación Automática del Plan de Imputación** 
Este bloque analiza los campos con valores faltantes detectados previamente y sugiere estrategias de manejo mediante reglas heurísticas:

1. **Filtrado de Nulos:** Se seleccionan únicamente las variables que presentan datos faltantes.
2. **Aplicación de Reglas:**
    * **Llaves/IDs:** Se identifican como críticos (no imputables).
    * **Nulos > 80%:** Se sugiere descartar la columna por falta de información.
    * **Nulos < 5%:** Se sugiere imputación estadística simple o eliminación de filas.
3. **Exportación:** Se genera el archivo 03_Plan_Imputacion.csv con la estrategia recomendada y su justificación.

In [ ]:
import polars as pl
from pathlib import Path

# 1. Cargar el Diccionario Exploratorio (Base del conocimiento)
archivo_exploratorio = Path("01_Diccionario_Exploratorio.csv")

try:
    if archivo_exploratorio.exists():
        df_exploratorio = pl.read_csv(archivo_exploratorio)
    else:
        raise FileNotFoundError
except FileNotFoundError:
    print(f"No se encontró el archivo {archivo_exploratorio}. Asegúrate de haberlo generado.")
    # Creación de DataFrame dummy para demostración si falta el archivo
    df_exploratorio = pl.DataFrame({
        "Archivo_Original": ["Econo", "Econo", "TS"],
        "Campo_Original": ["EXP", "RESULTADO", "EDAD"],
        "Cantidad_Nulos": [81, 1223, 0],
        "Porcentaje_Nulos": [1.75, 26.4, 0.0]
    })

print(f"Diccionario base cargado. Total campos: {df_exploratorio.height}")

# 2. Filtrar solo los campos que tienen Nulos
df_con_nulos = df_exploratorio.filter(pl.col("Cantidad_Nulos") > 0)

print(f"Campos con nulos detectados: {df_con_nulos.height}")

# 3. Definir y Aplicar Reglas Heurísticas
# Usamos pl.when().then() encadenados para reemplazar numpy.select
df_imputacion_export = df_con_nulos.with_columns([
    pl.when(
        # Caso 1: Llaves Primarias o IDs (Regex con (?i) para case-insensitive)
        pl.col("Campo_Original").str.contains("(?i)EXP|ID|FOLIO|HISTORIA")
    ).then(pl.lit("CRÍTICO: Imposible imputar (Llave Primaria). Revisar origen o descartar fila."))

    .when(
        # Caso 2: Vaciedad Extrema (> 80% nulos)
        pl.col("Porcentaje_Nulos") > 80.0
    ).then(pl.lit("Posible descarte de COLUMNA (Poca información)."))

    .when(
        # Caso 3: Vaciedad Leve (< 5%)
        pl.col("Porcentaje_Nulos") < 5.0
    ).then(pl.lit("Imputación Estadística Simple (Media/Moda) o Borrado de filas (listwise deletion)."))

    .otherwise(pl.lit("Requiere Análisis: Imputación avanzada (KNN/MICE) o modelo predictivo."))
    .alias("Estrategia_Imputacion"),

    # 4. Generar justificación automática
    pl.when(
        pl.col("Porcentaje_Nulos") > 80.0
    ).then(pl.lit("Alta tasa de valores faltantes reduce confiabilidad."))
    .otherwise(pl.lit(""))
    .alias("Justificacion")
]).select([
    # 5. Seleccionar columnas finales
    "Archivo_Original",
    "Campo_Original",
    "Cantidad_Nulos",
    "Porcentaje_Nulos",
    "Estrategia_Imputacion",
    "Justificacion"
])

# 6. Guardar
output_file = Path("03_Plan_Imputacion.csv")
df_imputacion_export.write_csv(output_file)

print(f"Archivo generado: {output_file}")
print("   Estrategias preliminares asignadas. Validar las sugerencias manualmente.")
display(df_imputacion_export)

## 4. Generación del Diccionario de Datos Final

1. Detecta el Bloque Semántico: Busca palabras clave (como "MUNICIPIO" o "DIAGNOSTICO") para asignarle el bloque automáticamente.

2. Propone Nombres Unificados: Toma el campo original sucio (ej. "FECHA DE NACIMIENTO") y lo convierte a snake_case limpio ("fecha_nacimiento").

3. Sugiere Tipos Estadísticos: Si Polars vio un número, sugiere "Numérica"; si vio texto, sugiere "Nominal" o "Texto Libre".

4. Crea la Trazabilidad: Genera la columna Origen_Datos para que sepas de dónde salió ese campo.

5. Exporta el Resultado: Guarda todo en `04_Diccionario_Objetivo.csv` listo para la siguiente fase del proyecto.

In [ ]:
import polars as pl

# 1. Cargar el Diccionario Exploratorio
try:
    df_exploratorio = pl.read_csv("01_Diccionario_Exploratorio.csv")
    print(f"Diccionario base cargado. Total campos: {df_exploratorio.height}")
except Exception as e:
    print(f"Error cargando 01_Diccionario_Exploratorio.csv: {e}")
    # Detenemos si no hay base
    df_exploratorio = pl.DataFrame()

if not df_exploratorio.is_empty():
    # 2. Definir Heurísticas para Bloques Semánticos (Usando Regex)
    # Orden importa: reglas específicas primero
    expr_bloque = (
        pl.when(pl.col("Campo_Original").str.contains("(?i)NOMBRE|PATERNO|MATERNO|SEXO|EDAD|NACIMIENTO|CURP|RFC|GENERO"))
        .then(pl.lit("Identidad Demográfica"))

        .when(pl.col("Campo_Original").str.contains("(?i)ESTADO|MUNICIPIO|PAIS|RESIDENCIA|DOMICILIO|GEO"))
        .then(pl.lit("Geográfico"))

        .when(pl.col("Campo_Original").str.contains("(?i)EXP|FOLIO|HISTORIA|INGRESO|EGRESO|ALTA|CAMA"))
        .then(pl.lit("Administrativo"))

        .when(pl.col("Campo_Original").str.contains("(?i)DIAG|CIE|COMORBI|SINTOMA|RESULTADO|MUESTRA|MOTIVO"))
        .then(pl.lit("Clínico"))

        .when(pl.col("Campo_Original").str.contains("(?i)OCUPACION|ESCOLARIDAD|NIVEL|SOCIO|ECONO|VIVIENDA"))
        .then(pl.lit("Socioeconómico"))

        .otherwise(pl.lit("Otros / Revisar"))
    )

    # 3. Definir Heurística para Nombres Unificados (Snake Case)
    # Ej: "FECHA DE NACIMIENTO" -> "fecha_nacimiento"
    expr_nombre_unificado = (
        pl.col("Campo_Original")
        .str.to_lowercase()
        .str.replace_all(r"[^a-z0-9]", "_") # Caracteres raros a guion bajo
        .str.replace_all(r"_+", "_")       # Eliminar guiones dobles
        .str.replace_all(r"^_{1}|_{1}$", "") # Trim de guiones
    )

    # 4. Definir Heurística para Tipo Variable Estadística
    expr_tipo_estadistico = (
        pl.when(pl.col("Tipo_Dato_Detectado").str.contains("Int|Float"))
        .then(pl.lit("Numérica (Continua/Discreta)"))
        .when(pl.col("Tipo_Dato_Detectado").str.contains("Date|Time"))
        .then(pl.lit("Temporal"))
        .otherwise(pl.lit("Nominal / Texto"))
    )

    # 5. Construir el DataFrame Objetivo
    df_objetivo = df_exploratorio.with_columns([
        expr_bloque.alias("Bloque_Semantico"),
        expr_nombre_unificado.alias("Nombre_Variable_Unificada"),
        pl.lit("").alias("Tipo_Dato_Final"), # Para que definas (VARCHAR, INT, DATE)
        expr_tipo_estadistico.alias("Tipo_Variable_Estadistica"),
        pl.lit("").alias("Descripcion_Semantica"), # ¡A llenar a mano!
        pl.format("{} -> {}", pl.col("Archivo_Original"), pl.col("Campo_Original")).alias("Origen_Datos")
    ])

    # 6. Seleccionar y Ordenar Columnas Finales
    df_objetivo_export = df_objetivo.select([
        "Bloque_Semantico",
        "Nombre_Variable_Unificada",
        "Tipo_Dato_Final",
        "Tipo_Variable_Estadistica",
        "Descripcion_Semantica",
        "Origen_Datos",
        "Ejemplo" # Útil tenerlo a la vista para describir
    ]).sort("Bloque_Semantico")

    # 7. Guardar
    output_file = "04_Diccionario_Datos_Objetivo.csv"
    df_objetivo_export.write_csv(output_file)

    print(f" Archivo generado: {output_file}")
    print("   Estructura unificada propuesta. Abre en LibreOffice y ajusta:")
    print("   1. Revisa si la asignación de 'Bloque' es correcta.")
    print("   2. Estandariza los 'Nombres Unificados' (que no haya duplicados).")
    print("   3. ¡Escribe las Descripciones Semánticas!")
    print("\nPrevisualización:")
    display(df_objetivo_export.select(["Bloque_Semantico", "Nombre_Variable_Unificada", "Origen_Datos"]))
else:
    print(" No se pudo generar el archivo porque el DataFrame base está vacío.")